# سیستم شباهت‌یابی مبتنی بر شبکه عصبی (پروژه کارآموزی)

این نوت‌بوک شامل پیاده‌سازی کامل و گام‌به‌گام فعالیت اجرایی کارآموزی است:
1. **ساخت پایگاه‌داده ۱۰۰ سطری:** هر سطر نماینده یک نمونه داده و دارای ۳ ویژگی برداری بازه‌ای است.
2. **تولید داده دریافتی (Query):** مقادیر عددی نقطه‌ای تصادفی از درون بازه‌ها استخراج می‌شوند (با ۱٪ نویز و ۳۰٪ شانس حذف عضو).
3. **حل گلوگاه ورودی با طول متغیر در شبکه عصبی:** استفاده از معماری پدینگ و ماسک اعتبار (Padding & Validity Masking) بدون سرچ روی دیتابیس و بدون خلاصه‌سازی آماری.
4. **استنتاج مستقیم با شبکه عصبی:** تعیین ۳ سطر با بیشترین شباهت و گزارش دقیق درصد احتمال آن‌ها.


In [ ]:
import numpy as np
import pandas as pd
from typing import List, Tuple, Dict, Any
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import json
import warnings

warnings.filterwarnings("ignore")
np.random.seed(42)
print("✅ تمام کتابخانه‌ها با موفقیت بارگذاری شدند.")


## مرحله اول: ساخت پایگاه‌داده ۱۰۰ سطری با ویژگی‌های بازه‌ای

مشخصات بازه‌ها مطابق تعریف کارآموزی:
- **Feature1:** بین ۱ تا ۱۰ بازه، حداکثر عرض بازه ۵، محدوده ۱۰۰۰ تا ۱۰۰۰۰
- **Feature2:** بین ۱ تا ۲۰ بازه، حداکثر عرض بازه ۱۰، محدوده ۲۰۰۰۰ تا ۱۰۰۰۰۰
- **Feature3:** بین ۱ تا ۵ بازه، حداکثر عرض بازه ۱، محدوده ۰.۱ تا ۱۰۰۰
- سقف همپوشانی بازه‌های سطرها با یکدیگر: ۳۰٪


In [ ]:
def generate_intervals(n_min: int, n_max: int, width_max: float, value_min: float, value_max: float, decimal_places: int = 2) -> List[Tuple[float, float]]:
    n = np.random.randint(n_min, n_max + 1)
    intervals = []
    for _ in range(n):
        start = np.random.uniform(value_min, value_max - width_max)
        width = np.random.uniform(0, width_max)
        end = start + width
        start, end = round(start, decimal_places), round(end, decimal_places)
        if start > end:
            start, end = end, start
        intervals.append((start, end))
    intervals.sort(key=lambda x: x[0])
    return intervals

def intervals_to_string(intervals: List[Tuple[float, float]]) -> str:
    return ",".join([f"{s}-{e}" for s, e in intervals])

def generate_database(n_rows: int = 100) -> pd.DataFrame:
    data = {"Feature1": [], "Feature2": [], "Feature3": []}
    for _ in range(n_rows):
        f1 = generate_intervals(1, 10, 5.0, 1000.0, 10000.0, 2)
        f2 = generate_intervals(1, 20, 10.0, 20000.0, 100000.0, 2)
        f3 = generate_intervals(1, 5, 1.0, 0.1, 1000.0, 3)
        data["Feature1"].append(intervals_to_string(f1))
        data["Feature2"].append(intervals_to_string(f2))
        data["Feature3"].append(intervals_to_string(f3))
    df = pd.DataFrame(data)
    df.index.name = "row_id"
    return df

df_database = generate_database(100)
df_database.to_pickle("database.pkl")
df_database.to_csv("database.csv", encoding="utf-8-sig")

print(f"✅ پایگاه‌داده ۱۰۰ سطری با موفقیت ایجاد و ذخیره شد. (تعداد سطرها: {len(df_database)})")
df_database.head()


## مرحله دوم: تولید داده دریافتی (Query) نقطه‌ای با طول متغیر

مطابق بازخورد کارفرما و مشخصات PDF:
- داده‌های دریافتی **بازه‌ای نیستند** و به صورت مقادیر عددی (نقاط) از درون بازه‌های یک سطر تصادفی انتخاب می‌شوند.
- با شانس ۳۰٪ ممکن است یکی از اعضای بردار مفقود شود (شبیه‌سازی افت داده در ارسال).
- به هر عدد ۱٪ خطای تصادفی اعمال می‌شود.


In [ ]:
def parse_intervals(interval_str: str) -> List[Tuple[float, float]]:
    if not interval_str or pd.isna(interval_str):
        return []
    intervals = []
    for part in str(interval_str).split(","):
        part = part.strip()
        if "-" in part:
            try:
                s, e = part.split("-")
                intervals.append((float(s), float(e)))
            except ValueError:
                continue
    return intervals

def generate_feature_points(interval_str: str, decimal_places: int = 4) -> Tuple[List[float], str]:
    intervals = parse_intervals(interval_str)
    if not intervals:
        return [], ""
    # ۱. انتخاب عدد تصادفی از درون هر بازه
    points = [round(float(np.random.uniform(s, e)), decimal_places) for s, e in intervals]
    # ۲. احتمال ۳۰٪ حذف یک عضو
    if len(points) > 1 and np.random.random() < 0.30:
        drop_idx = np.random.randint(0, len(points))
        points.pop(drop_idx)
    # ۳. اعمال ۱٪ نویز به هر عدد
    noisy_points = [round(float(p * (1.0 + np.random.uniform(-0.01, 0.01))), decimal_places) for p in points]
    points_str = ", ".join([f"{p:.{decimal_places}f}" for p in noisy_points])
    return noisy_points, points_str

def generate_query(df: pd.DataFrame) -> Dict:
    random_idx = int(np.random.randint(0, len(df)))
    row = df.iloc[random_idx]
    f1_pts, f1_str = generate_feature_points(row["Feature1"])
    f2_pts, f2_str = generate_feature_points(row["Feature2"])
    f3_pts, f3_str = generate_feature_points(row["Feature3"])
    return {
        "source_row_id": random_idx,
        "Feature1": f1_str,
        "Feature2": f2_str,
        "Feature3": f3_str,
        "Feature1_values": f1_pts,
        "Feature2_values": f2_pts,
        "Feature3_values": f3_pts,
    }

query_sample = generate_query(df_database)
with open("query.json", "w", encoding="utf-8") as f:
    json.dump(query_sample, f, ensure_ascii=False, indent=2)

print(f"🎯 سطر مبدأ انتخاب‌شده: سطر {query_sample['source_row_id']}")
print(f"Feature1 (اعداد نقطه‌ای): {query_sample['Feature1']}")
print(f"Feature2 (اعداد نقطه‌ای): {query_sample['Feature2'][:80]}...")
print(f"Feature3 (اعداد نقطه‌ای): {query_sample['Feature3']}")


## مرحله سوم: حل گلوگاه ورودی با طول متغیر در شبکه عصبی (Padding & Masking)

### رویکرد علمی بدون خلاصه آماری یا سرچ روی دیتابیس:
1. بردار داده‌های خام بر اساس کرانه‌های فیزیکی هر ویژگی به بازه $[0, 1]$ نرمال می‌شوند.
2. داده‌ها تا طول ماکزیمم هر ویژگی ($L_1=10, L_2=20, L_3=5$) با مقدار صفر پد می‌شوند.
3. یک بردار ماسک دودویی ($1$ برای موقعیت معتبر و $0$ برای پدینگ) همزمان وارد شبکه می‌شود.
4. نورون‌های لایه اول شبکه ضرایب تفکیک‌شده‌ای برای مقادیر و ماسک‌ها فرامی‌گیرند و موقعیت‌های پدشده را خنثی می‌کنند (Neural Gating).


In [ ]:
MAX_LEN_F1, MAX_LEN_F2, MAX_LEN_F3 = 10, 20, 5
MIN_VAL_F1, MAX_VAL_F1 = 1000.0, 10000.0
MIN_VAL_F2, MAX_VAL_F2 = 20000.0, 100000.0
MIN_VAL_F3, MAX_VAL_F3 = 0.1, 1000.0

def encode_feature_vector(points: List[float], max_len: int, min_val: float, max_val: float) -> Tuple[List[float], List[float]]:
    if not points:
        return [0.0] * max_len, [0.0] * max_len
    sorted_pts = sorted(points)
    span = max_val - min_val
    norm_pts = [(x - min_val) / span for x in sorted_pts]
    actual_len = len(norm_pts)
    pad_len = max(0, max_len - actual_len)
    values = norm_pts[:max_len] + [0.0] * pad_len
    mask = [1.0] * min(actual_len, max_len) + [0.0] * pad_len
    return values, mask

def encode_variable_length_query(q1: List[float], q2: List[float], q3: List[float]) -> np.ndarray:
    v1, m1 = encode_feature_vector(q1, MAX_LEN_F1, MIN_VAL_F1, MAX_VAL_F1)
    v2, m2 = encode_feature_vector(q2, MAX_LEN_F2, MIN_VAL_F2, MAX_VAL_F2)
    v3, m3 = encode_feature_vector(q3, MAX_LEN_F3, MIN_VAL_F3, MAX_VAL_F3)
    return np.array(v1 + v2 + v3 + m1 + m2 + m3, dtype=np.float32)

def generate_training_data(df: pd.DataFrame, samples_per_class: int = 60):
    parsed = [(parse_intervals(r["Feature1"]), parse_intervals(r["Feature2"]), parse_intervals(r["Feature3"])) for _, r in df.iterrows()]
    X, y = [], []
    for row_idx in range(len(df)):
        d1, d2, d3 = parsed[row_idx]
        for _ in range(samples_per_class):
            q1, _ = generate_feature_points(intervals_to_string(d1))
            q2, _ = generate_feature_points(intervals_to_string(d2))
            q3, _ = generate_feature_points(intervals_to_string(d3))
            X.append(encode_variable_length_query(q1, q2, q3))
            y.append(row_idx)
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.int32)

print("⏳ در حال تولید ۶۰۰۰ نمونه داده آموزشی برای ۱۰۰ سطر...")
X, y = generate_training_data(df_database, samples_per_class=60)
print(f"✅ ماتریس ورودی X با ابعاد {X.shape} (۳۵ مقدار نرمال‌شده + ۳۵ ماسک) ساخته شد.")

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

model = MLPClassifier(
    hidden_layer_sizes=(256, 128),
    activation="relu",
    solver="adam",
    learning_rate_init=0.001,
    max_iter=350,
    random_state=42,
    early_stopping=True,
    validation_fraction=0.10,
    n_iter_no_change=15
)

print("⏳ در حال آموزش شبکه عصبی عمیق روی ورودی‌های طول متغیر...")
model.fit(X_train, y_train)

test_probs = model.predict_proba(X_test)
test_acc = accuracy_score(y_test, model.predict(X_test))
top3_acc = np.mean([y_test[i] in np.argsort(test_probs[i])[-3:] for i in range(len(y_test))])

print(f"✅ دقت پیش‌بینی دقیق (Top-1 Accuracy): {test_acc * 100:.2f}%")
print(f"✅ دقت در ۳ سطر اول (Top-3 Accuracy): {top3_acc * 100:.2f}%")


## مرحله چهارم: استنتاج مستقیم شبکه عصبی و استخراج ۳ سطر با بیشترین شباهت

داده ورودی تنها در یک مرحله و بدون هیچ جستجو یا مقایسه‌ای در پایگاه‌داده، از شبکه عصبی عبور کرده و خروجی Softmax آن رتبه‌بندی ۳ سطر شبیه‌ترین را به همراه احتمال دقیق استخراج می‌کند.


In [ ]:
x_input = encode_variable_length_query(
    query_sample["Feature1_values"],
    query_sample["Feature2_values"],
    query_sample["Feature3_values"]
)

probs = model.predict_proba([x_input])[0]
top3_indices = np.argsort(probs)[-3:][::-1]

results = []
for rank, idx in enumerate(top3_indices, 1):
    row = df_database.iloc[idx]
    results.append({
        "رتبه": f"🥇 رتبه {rank}",
        "شماره سطر": int(idx),
        "درصد احتمال شباهت": f"{probs[idx] * 100:.2f}%",
        "Feature1 بازه‌ای": row["Feature1"][:45] + "...",
        "Feature2 بازه‌ای": row["Feature2"][:45] + "...",
        "Feature3 بازه‌ای": row["Feature3"]
    })

pd.DataFrame(results)
